# CS604 Colab launcher — Ana: where do the errors come from? (no training)

Scores two **pilot** checkpoints on their own pilot holdout (5 photos per class of 1,000 classes, never trained on), with `scripts/error_analysis.py`:

| Name | Run | Recipe |
|---|---|---|
| `lora` | `A_random_crops` of the training-crop pilot | LoRA r24, no extras (the current PEFT recipe) |
| `full_ft` | `F_full_ft_llrd` of the parameter-selection pilot | full fine-tuning, Van's recipe |

Both trained on the same pilot classes and fit photos, so their errors can be compared photo by photo. Full-scale checkpoints can't be used: they trained on the holdout photos.

Outputs (in `CS604_RUNS/error_analysis/`):
- `topk.png` and `summary.json`: top-1 to top-10, where the true class sits when the model is wrong, how many errors are relatives (within 5 class IDs);
- `overlap.json`: errors one model fixes for the other, and where the other ranked the answer;
- `errors_<name>.csv`: every error with its top-3 guesses and an empty `category` column to fill in by hand;
- `gallery_<name>_*.png`: 40 random errors per model, each with a training photo of the true and the guessed class.

About 25 min, mostly the data download; scoring is a few minutes. Any GPU works (inference only). Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
PILOT_DATA_DIR = "/content/pilot_data"
# Pilot runs to compare: (ablation folder, arm). Their run folders are read from the arm records.
RUNS = {
    "lora": ("ablation_ablation_train_crops", "A_random_crops"),
    "full_ft": ("ablation_ablation_param_selection", "F_full_ft_llrd"),
}
OUT_NAME = "error_analysis"
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Score both checkpoints on the pilot holdout

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from scripts.pilot_sweep import build_pilot_root

PILOT_ROOT, _ = build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 1000, 5, 0)  # the pilots' root: 1,000 classes, seed 0
run_args = []
for name, (folder, arm) in RUNS.items():
    record = Path(DRIVE_OUT) / folder / "arms" / f"{arm}.json"
    run_dir = json.loads(record.read_text())["run_dir"]
    assert (Path(run_dir) / "last.pt").is_file(), f"no checkpoint in {run_dir}"
    print(f"{name}: {arm} -> {run_dir}")
    run_args += ["--run", f"{name}={run_dir}"]
OUT = Path(DRIVE_OUT) / OUT_NAME
proc = subprocess.run([sys.executable, "-u", "scripts/error_analysis.py", "--pilot_root", str(PILOT_ROOT),
                       "--data_root", DATA_DIR, "--out_dir", str(OUT), *run_args])
if proc.returncode != 0:
    end_runtime(f"error analysis failed (exit {proc.returncode})")
    raise SystemExit("error analysis failed")


## 6. Results

- **Top-k curve:** the step from top-1 to top-2 is the share of photos whose answer was the second guess; the total up to top-5 is the room that choosing better among the model's own candidates could recover.
- **errors_guess_is_relative_within_5_ids:** how many errors are sibling confusions (same genus, roughly).
- **overlap:** what full FT gets right that LoRA misses, and where LoRA had ranked those answers.

Then open `errors_lora.csv` (Drive), go through the gallery images, and fill the `category` column: e.g. *sibling*, *tiny subject*, *odd life stage / part*, *not the organism*, *looks mislabelled*.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = json.loads((OUT / "summary.json").read_text())
rows = {name: {k: v for k, v in s.items() if not isinstance(v, dict) and k != "run_dir"} for name, s in summary.items() if name != "overlap"}
display(pd.DataFrame(rows))
display(pd.DataFrame({name: s["top_k"] for name, s in summary.items() if name != "overlap"}))
if "overlap" in summary:
    print(json.dumps(summary["overlap"], indent=2))
display(Image(filename=str(OUT / "topk.png")))
for png in sorted(OUT.glob("gallery_lora_*.png"))[:2]:
    display(Image(filename=str(png)))


## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")